[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abuhuzaifahbidin/megat/blob/main/notebooks/01_download_data.ipynb)

# Notebook 01 — Download Malay Corpus

**What this notebook does:**
1. Installs required packages
2. Mounts Google Drive (so data survives if the Colab session dies)
3. Downloads ~4 million documents from FineWeb2 Malay (`zsm_Latn`)
4. Optionally adds Malay Wikipedia
5. Spot-checks the data quality
6. Saves everything to Google Drive

**Expected runtime:** 2–4 hours (streaming download, depends on Colab network speed)

**Expected output:** `fineweb2_malay.jsonl` (~5–7 GB) saved to your Drive

---
**After this notebook, run:** `02_train_tokenizer.ipynb`

## Step 0 — Install packages

In [1]:
# These are not pre-installed on Colab
!pip install -q datasets huggingface_hub

## Step 1 — Mount Google Drive

We save everything to Drive so that if the Colab session disconnects mid-download,
you don't lose progress — you can resume from where you left off.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os

# All outputs from this notebook go here.
# Change this path if you want to save somewhere else in your Drive.
SAVE_DIR = '/content/drive/MyDrive/megat_data'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Saving to: {SAVE_DIR}')

Mounted at /content/drive
Saving to: /content/drive/MyDrive/megat_data


## Step 2 — Download FineWeb2 Malay

**Why FineWeb2?**
FineWeb2 is HuggingFace's cleaned, deduplicated web crawl dataset. The Malay subset
(`zsm_Latn`) contains ~9.46 million documents — already filtered for quality using
GlotLID language ID, MinHash deduplication, and heuristic quality filters.
It outperforms mC4, OSCAR, and CulturaX on downstream benchmarks.

**Why streaming?**
The full dataset is too large to download all at once. Streaming iterates over documents
one at a time, writing them to disk immediately. This way:
- You only use a few MB of RAM regardless of corpus size
- You can stop at any target document count (we want ~4M docs)
- If the session dies, you keep everything already saved

**Why ~4M documents?**
Each doc averages ~500–600 tokens with our Malay tokenizer.
4M docs × 550 tokens ≈ 2.2 billion tokens — enough for 3 epochs during the 72-hour run.

In [3]:
# ── Configuration ──────────────────────────────────────────────────────────────
TARGET_DOCS   = 4_000_000   # number of documents to download
SAVE_EVERY    = 100_000     # write progress to disk every N docs (in case of disconnect)
OUTPUT_FILE   = os.path.join(SAVE_DIR, 'fineweb2_malay.jsonl')

print(f'Target docs:  {TARGET_DOCS:,}')
print(f'Output file:  {OUTPUT_FILE}')

Target docs:  4,000,000
Output file:  /content/drive/MyDrive/megat_data/fineweb2_malay.jsonl


In [4]:
import json
import time
from datasets import load_dataset

# Check if we already have a partial download so we can resume.
# We count existing lines in the output file and skip that many docs.
already_saved = 0
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        already_saved = sum(1 for _ in f)
    print(f'Resuming from doc {already_saved:,} (found existing file)')
else:
    print('Starting fresh download...')

if already_saved >= TARGET_DOCS:
    print(f'Already have {already_saved:,} docs — nothing to do!')
else:
    # streaming=True means documents are fetched one at a time, not all at once.
    # This is critical — the full dataset is tens of GBs.
    print('Loading FineWeb2 Malay stream...')
    ds = load_dataset(
        'HuggingFaceFW/fineweb-2',
        'zsm_Latn',
        split='train',
        streaming=True
    )

    count    = already_saved
    t_start  = time.time()
    t_last   = t_start

    with open(OUTPUT_FILE, 'a') as f:          # 'a' = append so we don't overwrite existing
        for example in ds.skip(already_saved): # skip docs we already have
            # We only save the text field — we don't need metadata for training.
            f.write(json.dumps({'text': example['text']}) + '\n')
            count += 1

            # Print progress every SAVE_EVERY docs
            if count % SAVE_EVERY == 0:
                now      = time.time()
                rate     = SAVE_EVERY / (now - t_last)   # docs/sec
                remaining = (TARGET_DOCS - count) / rate / 60  # minutes
                t_last   = now
                size_gb  = os.path.getsize(OUTPUT_FILE) / 1e9
                print(f'  [{count:>7,} / {TARGET_DOCS:,}]  rate: {rate:.0f} docs/s  '
                      f'ETA: {remaining:.0f} min  file: {size_gb:.2f} GB')

            if count >= TARGET_DOCS:
                break

    total_time = (time.time() - t_start) / 60
    file_size  = os.path.getsize(OUTPUT_FILE) / 1e9
    print(f'\nDone!  {count:,} docs saved  |  {file_size:.2f} GB  |  {total_time:.1f} min')

Starting fresh download...
Loading FineWeb2 Malay stream...


README.md: 0.00B [00:00, ?B/s]

  [100,000 / 4,000,000]  rate: 955 docs/s  ETA: 68 min  file: 0.38 GB
  [200,000 / 4,000,000]  rate: 1100 docs/s  ETA: 58 min  file: 0.74 GB
  [300,000 / 4,000,000]  rate: 995 docs/s  ETA: 62 min  file: 1.10 GB
  [400,000 / 4,000,000]  rate: 1063 docs/s  ETA: 56 min  file: 1.46 GB
  [500,000 / 4,000,000]  rate: 978 docs/s  ETA: 60 min  file: 1.82 GB
  [600,000 / 4,000,000]  rate: 1103 docs/s  ETA: 51 min  file: 2.18 GB
  [700,000 / 4,000,000]  rate: 1117 docs/s  ETA: 49 min  file: 2.53 GB
  [800,000 / 4,000,000]  rate: 981 docs/s  ETA: 54 min  file: 2.87 GB
  [900,000 / 4,000,000]  rate: 1019 docs/s  ETA: 51 min  file: 3.23 GB
  [1,000,000 / 4,000,000]  rate: 1005 docs/s  ETA: 50 min  file: 3.57 GB
  [1,100,000 / 4,000,000]  rate: 1051 docs/s  ETA: 46 min  file: 3.92 GB
  [1,200,000 / 4,000,000]  rate: 1027 docs/s  ETA: 45 min  file: 4.26 GB
  [1,300,000 / 4,000,000]  rate: 1055 docs/s  ETA: 43 min  file: 4.60 GB
  [1,400,000 / 4,000,000]  rate: 1092 docs/s  ETA: 40 min  file: 4.94 GB


## Step 3 — Add Malay Wikipedia (optional but recommended)

Wikipedia is high-quality, well-structured prose — encyclopedic articles written
in formal Malay. It adds ~200–400 MB of clean text that complements the more
varied, informal style of FineWeb2 web crawl data.

Skip this if you're short on time — FineWeb2 alone is sufficient.

In [7]:
import os
import json
from datasets import load_dataset

WIKI_FILE   = os.path.join(SAVE_DIR, 'malay_wikipedia.jsonl')
ADD_WIKI    = True  # set to False to skip

if ADD_WIKI:
    if os.path.exists(WIKI_FILE):
        print('Wikipedia already downloaded — skipping.')
    else:
        print('Downloading Malay Wikipedia (20231101.ms)...')

        # Switched to the new 'wikimedia/wikipedia' path and removed trust_remote_code
        wiki = load_dataset(
            'wikimedia/wikipedia',
            '20231101.ms',
            split='train'
        )

        with open(WIKI_FILE, 'w', encoding='utf-8') as f:
            for article in wiki:
                # Standardizing to JSONL format
                f.write(json.dumps({'text': article['text']}, ensure_ascii=False) + '\n')

        wiki_size = os.path.getsize(WIKI_FILE) / 1e6
        print(f'Wikipedia saved: {len(wiki):,} articles  |  {wiki_size:.0f} MB')
else:
    print('Skipping Wikipedia (ADD_WIKI = False).')

README.md: 0.00B [00:00, ?B/s]

20231101.ms/train-00000-of-00001.parquet:   0%|          | 0.00/210M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/368628 [00:00<?, ? examples/s]

Wikipedia saved: 368,628 articles  |  401 MB


## Step 4 — Spot-check data quality

FineWeb2 is already heavily cleaned, but it's worth sampling a few hundred docs
to sanity-check before spending 72 hours training on it.

Look for:
- Documents that are actually English (language filter miss)
- Very short or near-empty documents
- HTML artifacts (`&amp;`, `<p>`, etc.)
- Garbled/encoding-broken text

In [8]:
import random

# Load a random sample of 200 documents for inspection
SAMPLE_SIZE = 200
all_lines   = []

with open(OUTPUT_FILE, 'r') as f:
    for i, line in enumerate(f):
        if i < SAMPLE_SIZE * 50:        # read first 10K docs
            all_lines.append(line)

sample = random.sample(all_lines, min(SAMPLE_SIZE, len(all_lines)))
docs   = [json.loads(line)['text'] for line in sample]

# Basic stats
lengths    = [len(d.split()) for d in docs]
avg_words  = sum(lengths) / len(lengths)
short_docs = sum(1 for l in lengths if l < 30)

print(f'Sample size:      {len(docs)} documents')
print(f'Avg word count:   {avg_words:.0f} words/doc')
print(f'Very short (<30): {short_docs} docs ({short_docs/len(docs)*100:.1f}%)')
print()

Sample size:      200 documents
Avg word count:   591 words/doc
Very short (<30): 0 docs (0.0%)



In [9]:
# Print 5 random documents in full for manual inspection
# Read these carefully — does it look like natural Malay?

for i, doc in enumerate(random.sample(docs, 5)):
    print(f'─── Document {i+1} ───────────────────────────────────────────')
    print(doc[:800])  # first 800 characters
    print()

─── Document 1 ───────────────────────────────────────────
Hotel mewah lima bintang ini yang menjadi penerima anugerah oleh Michelin terletak di jantung Kowloon, memudahkan kunjungan ke Hong Kong. Untuk ke pusat bandar Hong Kong adalah mudah kerana stesen keretapai bawah tanah Mongkok East MTR terletak di bawah Royal Plaza Hotel.Hotel ini memiliki bilik tetamu yang serba lengkap dan luas. Kompleks membeli-belah yang bersambung dengan hotel ini menawarkan pelbagai pilihan kedai dan restoran. Royal Plaza Hotel juga berdekatan dengan Pasar Burung, Pasar Bunga, Pasar Ikan Emas dan Pasar Wanita yang terkenal. Perkhidmatan bas percuma beroperasi di antara hotel ini dengan daerah tumpuan pelancong Tsim Sha Tsui. Tetamu yang menggunakan Lapangan Terbang Hong Kong tidak perlu gusar kerana terdapat perkhidmatan bas Airport Express Shuttle Bus berdekatan hot

─── Document 2 ───────────────────────────────────────────
|Hj Sulaiman Abdullah adalah peguam yang dihormati kawan dan lawan kerana prinsi

In [10]:
# Quick automated checks — flag anything suspicious

english_indicators = ['the ', 'and ', 'that ', ' is ', ' are ', ' was ']
html_indicators    = ['<p>', '<div>', '&amp;', '&lt;', 'href=']

flagged_english = 0
flagged_html    = 0

for doc in docs:
    doc_lower = doc.lower()
    # Count English function word matches (crude but fast)
    english_hits = sum(doc_lower.count(w) for w in english_indicators)
    malay_hits   = doc_lower.count('yang ') + doc_lower.count('dan ') + doc_lower.count('untuk ')
    if english_hits > malay_hits * 2:
        flagged_english += 1
    if any(tag in doc_lower for tag in html_indicators):
        flagged_html += 1

print(f'Possibly English:   {flagged_english} / {len(docs)} ({flagged_english/len(docs)*100:.1f}%)')
print(f'HTML artifacts:     {flagged_html} / {len(docs)} ({flagged_html/len(docs)*100:.1f}%)')
print()
print('If English > 5% or HTML > 10%, consider a stricter filter pass.')
print('FineWeb2 is usually clean — <2% false positives is normal and acceptable.')

Possibly English:   2 / 200 (1.0%)
HTML artifacts:     0 / 200 (0.0%)

If English > 5% or HTML > 10%, consider a stricter filter pass.
FineWeb2 is usually clean — <2% false positives is normal and acceptable.


## Step 5 — Verify save

Final check before moving to the tokenizer notebook.

In [11]:
# Count total lines in the final file
total_lines = 0
with open(OUTPUT_FILE, 'r') as f:
    for _ in f:
        total_lines += 1

file_size_gb = os.path.getsize(OUTPUT_FILE) / 1e9

print('── Final file summary ─────────────────────────────')
print(f'  Path:       {OUTPUT_FILE}')
print(f'  Documents:  {total_lines:,}')
print(f'  Size:       {file_size_gb:.2f} GB')

if ADD_WIKI and os.path.exists(WIKI_FILE):
    wiki_size = os.path.getsize(WIKI_FILE) / 1e6
    print(f'  Wikipedia:  {WIKI_FILE} ({wiki_size:.0f} MB)')

print()
print('Next: run 02_train_tokenizer.ipynb')

── Final file summary ─────────────────────────────
  Path:       /content/drive/MyDrive/megat_data/fineweb2_malay.jsonl
  Documents:  4,000,000
  Size:       13.68 GB
  Wikipedia:  /content/drive/MyDrive/megat_data/malay_wikipedia.jsonl (401 MB)

Next: run 02_train_tokenizer.ipynb
